In [36]:
import sys
import os

# Add the parent directory to the path so the package is importable
sys.path.append(os.path.abspath(".."))

from llm_data_quality_assistant.pipeline import Pipeline
from llm_data_quality_assistant.enums import Models, CorruptionTypes
import pandas as pd
from pprint import pprint
from dotenv import load_dotenv
import numpy as np
import jupyter_helper_functions
import string
import time

load_dotenv()

True

In [37]:
gold_standard = pd.read_csv(
    "../datasets/parker_datasets/allergen/allergen_cleaned_gold_first1000.csv"
)
corrupted_dataset = jupyter_helper_functions.load_dataset(
    "../datasets/parker_datasets/allergen_custom/allergen_custom_corruption.csv"
)
corrupted_dataset = corrupted_dataset.fillna(0)
corrupted_dataset = corrupted_dataset.astype(int)
print(corrupted_dataset)

output = Pipeline.standardize_datasets("code", gold_standard=gold_standard, corrupted_dataset = corrupted_dataset)
gold_standard = output["gold_standard"]
corrupted_dataset = output["corrupted_dataset"]

# Duplicate and append the DataFrame 5 times
# corrupted_versions = 5
# gold_standard_extended = pd.concat([gold_standard.copy() for _ in range(corrupted_versions)], ignore_index=True)
# gold_standard_extended = pd.concat([group for _, group in gold_standard_extended.groupby("dicom_uid")], ignore_index=True)

              code  nuts  almondnuts  brazil_nuts  macadamia_nuts  hazelnut  \
0    4104420006065     0           0            0               0         0   
1    4104420006065     0           0            0               0         0   
2    4104420007963     2           1            0               0         2   
3    4104420007963     2           1            0               0         2   
4    4104420007987     0           2            0               0         2   
..             ...   ...         ...          ...             ...       ...   
201       42256199     1           1            0               0         0   
202       42330660     0           0            0               0         0   
203       42330660     0           0            0               0         0   
204       42373186     1           1            0               0         0   
205       42373186     1           1            0               0         0   

     pistachio  walnut  cashew  celery  ...  fish  

In [38]:
# corrupted_dataset = Pipeline.generate_corrupted_datasets(
#     dataset=gold_standard,
#     cell_corruption_types=[CorruptionTypes.CellCorruptionTypes.NULL],
#     row_corruption_types=[],
#     columns_to_exclude=["code"],
#     inplace=True,
#     severity=0.15,
#     output_size=1
# )
# corrupted_dataset = corrupted_dataset[0]
# jupyter_helper_functions.save_dataframe_csv(corrupted_dataset, "../datasets/parker_datasets/allergen_custom/allergen_custom_corruption.csv")
# raise ValueError("dlfsjk")

In [ ]:
rpm = 0
model_name = Models.OpenAIModels.GPT_4_1
context_rows = 0
file_name = jupyter_helper_functions.sanitize_filename(f"{model_name.value}_{context_rows}_rows_context")
primary_key = "code"

additional_context = f"""
{corrupted_dataset.sample(n=context_rows).to_string(index=False)}
"""

# Merge/clean with LLM
merged_df, time_taken = jupyter_helper_functions.merge_with_llm_timed(
    dataset=corrupted_dataset,
    primary_key=primary_key,
    model=model_name,
    rpm=rpm,
    additional_prompt=additional_context
)


Merging groups with LLM: 100%|██████████| 103/103 [03:18<00:00,  1.93s/it]


In [40]:
jupyter_helper_functions.save_dataframe_csv(merged_df, f"../analysis/repairs/allergen_custom/{file_name}_repair.csv")

In [41]:
import json


# Evaluate results
jupyter_helper_functions.standardize_and_evaluate(
    gold_standard=gold_standard,
    merged_df=merged_df,
    corrupt_dataset=corrupted_dataset,
    primary_key=primary_key,
    time_delta=time_taken,
    results_dir=f"../analysis/results/allergen_custom/",
    file_name=file_name,
)

{'accuracy': 0.9902912621359223,
 'column_names': ['code',
                  'nuts',
                  'almondnuts',
                  'brazil_nuts',
                  'macadamia_nuts',
                  'hazelnut',
                  'pistachio',
                  'walnut',
                  'cashew',
                  'celery',
                  'crustaceans',
                  'eggs',
                  'fish',
                  'gluten',
                  'lupin',
                  'milk',
                  'molluscs',
                  'mustard',
                  'peanut',
                  'sesame',
                  'soy',
                  'sulfite'],
 'f1_score': 0.725,
 'false_negative': 31,
 'false_negative_rate': 0.34831460674157305,
 'false_positive': 13,
 'false_positive_rate': 0.002925950934053567,
 'num_columns': 22,
 'num_rows': 206,
 'precision': 0.8169014084507042,
 'recall': 0.651685393258427,
 'time_taken': 198.95347380638123,
 'true_negative': 4430,
 'true_positive